# Parallel phylogenetic likelihood: CUDA validation and benchmark

Start a fresh Kaggle notebook with an NVIDIA GPU and Internet access. Until the repositories are public, run `scripts/package_notebook_sources.sh`, upload the resulting ZIP as a Kaggle dataset, and attach it to this notebook. The run validates the host implementation, CPU CUDA-algebra emulation, native CUDA kernels, and (when available) Compute Sanitizer before timing synthetic scaling cases and the public 11,638-taxon Fish Tree of Life alignment. JIT compilation is not involved.

In [ ]:
%%bash
set -euo pipefail

work=/kaggle/working/parallel-tree-inference
bundle=$(find /kaggle/input -type f -name 'parallel_tree_inference_sources.zip' -print -quit || true)
unpacked=$(find /kaggle/input -type d -name 'parallel_phylogenetic_inference' -print -quit || true)
if [[ -n "${bundle}" ]]; then
  rm -rf "${work}"
  mkdir -p "${work}"
  unzip -q "${bundle}" -d "${work}"
elif [[ -n "${unpacked}" ]]; then
  source_root=$(dirname "${unpacked}")
  rm -rf "${work}"
  mkdir -p "${work}"
  for repo in bidirectional_tree_rake_compress parallel_tree_hmm parallel_phylogenetic_inference; do
    if [[ ! -d "${source_root}/${repo}" ]]; then
      echo "the attached input is missing ${repo}" >&2
      exit 2
    fi
    cp -R "${source_root}/${repo}" "${work}/${repo}"
  done
else
  echo "attach the parallel_tree_inference_sources input before running" >&2
  find /kaggle/input -maxdepth 3 -print >&2
  exit 2
fi

cd "${work}/parallel_phylogenetic_inference"
TREE_HMM_BENCHMARK_REPEATS=5 \
  bash scripts/notebook_cuda.sh 2>&1 | \
  tee /kaggle/working/parallel_phylogenetics_cuda_report.txt


The complete machine description, validation log, and benchmark CSV rows are saved as `/kaggle/working/parallel_phylogenetics_cuda_report.txt`. Download that file after the run.